In [13]:
import json

with open('labeled.json', 'r', encoding='utf-8') as f:
    tasks = json.load(f)

few_shot_examples = []
for task in tasks[:15]:  # 只取前 15 张
    text = task['data']['text']
    entities = {}
    if task.get('annotations'):
        for res in task['annotations'][0]['result']:
            if 'labels' in res['value']:
                label = res['value']['labels'][0]  # 如 "B-INVOICE_NUMBER"
                span = res['value']['text']
                # 简单合并同类实体（demo 够用）
                key = label.replace('B-', '').replace('I-', '')
                entities[key] = entities.get(key, '') + ' ' + span
    few_shot_examples.append({
        "input_text": text[:300] + "..." if len(text) > 300 else text,
        "output": entities
    })

# 保存或打印
with open('few_shot_examples.json', 'w', encoding='utf-8') as f:
    json.dump(few_shot_examples, f, ensure_ascii=False, indent=2)

print("已生成 few-shot 示例")

test_example = []
with open('invoices.json', 'r', encoding='utf-8') as f:
    test_example = json.load(f)
    
print(f"\n loaded test example: {test_example[14:]}")
    

已生成 few-shot 示例

 loaded test example: [{'data': {'text': 'Yukon Packing INVOICE NO. CA-001 443 Maple Avenue Ontario, NT B4M 3B7 BILL To SHIP To INVOICE DATE 29/01/2019 Alferd Griner Packer Alferd Griner Packer P.0.# 1630/2019 765 Polar Bear Ave 185 Red River Ave Due DATE 26/04/2019 Vancouver; AB T4 Burnaby, NT 281 QtY DESCRIPTION UNIT PRICE AMOUNT Smoked chinook salmon fillet 100.00 100.00 Maple bacon doughnuts 15.00 30.00 Poutine curds 5.00 15.00 Subtolal 145.00 GST 5.0% 7.25 Total S152.25 A XMh Terms Conditions Payment is due within 15 days Please make cheques payable to: Yukon Packing'}}, {'data': {'text': 'Yukon Packing 443 Maple Avenue Ontario, NT B4M 3B7 BILL TO SHIP TO INVOICE # CA-001 Alferd Griner Packer Alferd Griner Packer INVOICE DATE 29/01/2019 765 Polar Bear Ave 185 Red River Ave P.o# 1630/2019 Vancouver, AB T4 Burnaby, NT 281 DUE DATE 26/04/2019 Invoice Total S152.25 QTy DESCRIPTION UNIT PRICE AMOUNT Smoked chinook salmon filllet 100.00 100.00 Maple bacon doughnuls 15.0

In [22]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

example_output = {"INVOICE_NUMBER": "CA-001", "INVOICE_DATE": "2019-01-29", "DUE_DATE": "2019-04-26", "VENDOR_NAME": "Yukon Packing", "TOTAL_AMOUNT": 152.25}

system_prompt = """
        You are a professional invoice extraction assistant. Only output valid JSON, no explanations.

        extract the following fields from the invoice text：
        - INVOICE_NUMBER
        - INVOICE_DATE (YYYY-MM-DD or DD/MM/YYYY) 
        - DUE_DATE (YYYY-MM-DD or DD/MM/YYYY)
        - VENDOR_NAME
        - TOTAL_AMOUNT


        example1：
        input_text：{example1_input_text}
        output：{example_output}
        
        example2：
        input_text：{example2_input_text}
        output：{example2_output}
        
        example3：
        input_text：{example3_input_text}
        output：{example3_output}
        
        example4：
        input_text：{example4_input_text}
        output：{example4_output}
        
        example5：
        input_text：{example5_input_text}
        output：{example5_output}


        Extract the fields from the human input invoice text

        """
        
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
        ("human", "Extract the fields from the following invoice text: {input_text}")
])

In [15]:
from pydantic import BaseModel, Field

class invoice_data(BaseModel):
    INVOICE_NUMBER: str = Field(..., description="INVOICE_NUMBER")
    INVOICE_DATE: str = Field(..., description="INVOICE_DATE，format: YYYY-MM-DD or DD/MM/YYYY")
    DUE_DATE: str = Field(..., description="DUE_DATE，format: YYYY-MM-DD or DD/MM/YYYY")
    VENDOR_NAME: str = Field(..., description="VENDOR_NAME")
    TOTAL_AMOUNT: float = Field(..., description="TOTAL_AMOUNT")
    


In [ ]:

from dotenv import load_dotenv
import os
load_dotenv()

# 1. in-order to present the demo, I have to use online LLM to process the pipeline. In the real case, we will use local LLM.
llm = init_chat_model(
        "gemini-2.5-flash",               
        model_provider="google_genai",
        google_api_key=os.getenv("GEMINI_API_KEY"),
        temperature=0.0,
        max_output_tokens=8192,
        #  thinking_level="low"（new attribute in new version）
    )

 # 2. chain
chain = (
    # assign data to the prompt, and convert dict to json string
    RunnablePassthrough.assign(
        example1_input_text=lambda _: few_shot_examples[0]['input_text'],
        example_output=lambda _: json.dumps(few_shot_examples[0]['output'], ensure_ascii=False, indent=2),
        example2_input_text=lambda _: few_shot_examples[1]['input_text'], 
        example2_output=lambda _: json.dumps(few_shot_examples[1]['output'], ensure_ascii=False, indent=2),
        example3_input_text=lambda _: few_shot_examples[2]['input_text'],
        example3_output=lambda _: json.dumps(few_shot_examples[2]['output'], ensure_ascii=False, indent=2),
        example4_input_text=lambda _: few_shot_examples[3]['input_text'],
        example4_output=lambda _: json.dumps(few_shot_examples[3]['output'], ensure_ascii=False, indent=2),
        example5_input_text=lambda _: few_shot_examples[4]['input_text'],
        example5_output=lambda _: json.dumps(few_shot_examples[4]['output'], ensure_ascii=False, indent=2),
        input_text=lambda x: json.dumps(x["metrics"], ensure_ascii=False, indent=2)
    )
    | prompt
    | llm.with_structured_output(invoice_data)
)


model_result = chain.invoke({"metrics": test_example[6]['data']['text']})

print(model_result)

INVOICE_NUMBER='CA-001' INVOICE_DATE='29/01/2019' DUE_DATE='26/04/2019' VENDOR_NAME='Yukon Packing' TOTAL_AMOUNT=152.25
